# G-Code Embedding Fine-Tuning: Gun Part Classifier

Hybrid approach: statistical features from g-code + text embeddings from sampled g-code lines.

**Data layout:**
- `../data-prep/g-code-spliced/`, g-code files (gun parts, label=1)
- `../data-prep/stl-manifest.json`, metadata/labels for the above
- `../data-prep/g-code-non-gun/`, (future) non-gun g-code files (label=0)

In [ ]:
!pip install torch sentence-transformers scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Configuration

In [ ]:
DATA_DIR = Path("../data-prep/g-code-spliced")
NON_GUN_DIR = Path("../data-prep/g-code-non-gun")  # add non-gun g-codes here later
MANIFEST_PATH = Path("../data-prep/stl-manifest.json")

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # 384-dim, fast, good baseline
TEXT_SAMPLE_LINES = 128  # lines sampled from each g-code for the text branch
MAX_TEXT_LENGTH = 512    # max tokens for the embedding model
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
EPOCHS = 20
VAL_SPLIT = 0.2
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Load Manifest & Discover Files

In [ ]:
with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

# Build list: (gcode_path, label, metadata)
samples = []

# Gun parts from manifest (label=1)
for entry in manifest:
    gcode_file = DATA_DIR / entry["gcode"]
    if gcode_file.exists():
        samples.append({
            "path": str(gcode_file),
            "label": 1,
            "category": entry.get("category", ""),
            "part": entry.get("part", ""),
            "labels": entry.get("labels", []),
        })

# Non-gun parts (label=0), add when available
if NON_GUN_DIR.exists():
    for gcode_file in sorted(NON_GUN_DIR.glob("*.gcode")):
        samples.append({
            "path": str(gcode_file),
            "label": 0,
            "category": "non-gun",
            "part": gcode_file.stem,
            "labels": [],
        })

df = pd.DataFrame(samples)
print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nCategories: {df['category'].unique()}")

## 3. G-Code Feature Extraction

Parse each g-code file into a fixed-length numerical feature vector capturing:
- Header metadata (print time, filament, bounding box, layer count/height)
- Movement statistics (G0 vs G1 counts, travel vs extrusion ratio)
- Coordinate statistics (X/Y/Z ranges, centroids, spread)
- Speed/feed rate statistics
- Extrusion statistics (total, rate distribution)

In [ ]:
def extract_gcode_features(filepath: str) -> dict:
    """Extract numerical features from a g-code file."""
    with open(filepath, "r", errors="ignore") as f:
        lines = f.readlines()

    total_lines = len(lines)

    # Header metadata
    header_vals = {
        "time": 0, "filament_m": 0, "layer_height": 0,
        "minx": 0, "miny": 0, "minz": 0,
        "maxx": 0, "maxy": 0, "maxz": 0,
        "layer_count": 0,
    }
    for line in lines[:50]:  # header is in first ~30 lines
        line_u = line.strip().upper()
        if line_u.startswith(";TIME:"):
            try: header_vals["time"] = float(line.strip().split(":")[1])
            except: pass
        elif ";FILAMENT USED:" in line_u:
            m = re.search(r"([\d.]+)", line.strip().split(":")[1])
            if m: header_vals["filament_m"] = float(m.group(1))
        elif ";LAYER HEIGHT:" in line_u:
            try: header_vals["layer_height"] = float(line.strip().split(":")[1])
            except: pass
        elif line_u.startswith(";MINX:"):
            try: header_vals["minx"] = float(line.strip().split(":")[1])
            except: pass
        elif line_u.startswith(";MINY:"):
            try: header_vals["miny"] = float(line.strip().split(":")[1])
            except: pass
        elif line_u.startswith(";MINZ:"):
            try: header_vals["minz"] = float(line.strip().split(":")[1])
            except: pass
        elif line_u.startswith(";MAXX:"):
            try: header_vals["maxx"] = float(line.strip().split(":")[1])
            except: pass
        elif line_u.startswith(";MAXY:"):
            try: header_vals["maxy"] = float(line.strip().split(":")[1])
            except: pass
        elif line_u.startswith(";MAXZ:"):
            try: header_vals["maxz"] = float(line.strip().split(":")[1])
            except: pass
        elif ";LAYER_COUNT:" in line_u:
            try: header_vals["layer_count"] = int(line.strip().split(":")[1])
            except: pass

    # Movement parsing
    g0_count = 0
    g1_count = 0
    x_vals, y_vals, z_vals = [], [], []
    e_vals = []
    f_vals = []

    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith(";"):
            continue

        if stripped.startswith("G0 ") or stripped.startswith("G0\t"):
            g0_count += 1
        elif stripped.startswith("G1 ") or stripped.startswith("G1\t"):
            g1_count += 1

        # Parse coordinate params
        parts = stripped.split(";")[0].split()  # remove inline comments
        for p in parts:
            try:
                if p.startswith("X"): x_vals.append(float(p[1:]))
                elif p.startswith("Y"): y_vals.append(float(p[1:]))
                elif p.startswith("Z"): z_vals.append(float(p[1:]))
                elif p.startswith("E"): e_vals.append(float(p[1:]))
                elif p.startswith("F"): f_vals.append(float(p[1:]))
            except ValueError:
                pass

    def safe_stats(vals):
        if not vals:
            return 0, 0, 0, 0
        arr = np.array(vals)
        return float(arr.min()), float(arr.max()), float(arr.mean()), float(arr.std())

    x_min, x_max, x_mean, x_std = safe_stats(x_vals)
    y_min, y_max, y_mean, y_std = safe_stats(y_vals)
    z_min, z_max, z_mean, z_std = safe_stats(z_vals)
    e_min, e_max, e_mean, e_std = safe_stats(e_vals)
    f_min, f_max, f_mean, f_std = safe_stats(f_vals)

    total_moves = g0_count + g1_count
    extrusion_ratio = g1_count / max(total_moves, 1)

    features = {
        # Header
        "print_time": header_vals["time"],
        "filament_m": header_vals["filament_m"],
        "layer_height": header_vals["layer_height"],
        "layer_count": header_vals["layer_count"],
        "header_minx": header_vals["minx"],
        "header_miny": header_vals["miny"],
        "header_minz": header_vals["minz"],
        "header_maxx": header_vals["maxx"],
        "header_maxy": header_vals["maxy"],
        "header_maxz": header_vals["maxz"],
        # File stats
        "total_lines": total_lines,
        "g0_count": g0_count,
        "g1_count": g1_count,
        "total_moves": total_moves,
        "extrusion_ratio": extrusion_ratio,
        # Coordinate stats
        "x_min": x_min, "x_max": x_max, "x_mean": x_mean, "x_std": x_std,
        "x_range": x_max - x_min,
        "y_min": y_min, "y_max": y_max, "y_mean": y_mean, "y_std": y_std,
        "y_range": y_max - y_min,
        "z_min": z_min, "z_max": z_max, "z_mean": z_mean, "z_std": z_std,
        "z_range": z_max - z_min,
        # Extrusion stats
        "e_min": e_min, "e_max": e_max, "e_mean": e_mean, "e_std": e_std,
        # Feed rate stats
        "f_min": f_min, "f_max": f_max, "f_mean": f_mean, "f_std": f_std,
    }
    return features

# Test on one file
test_features = extract_gcode_features(str(DATA_DIR / "1.gcode"))
print(f"Feature count: {len(test_features)}")
for k, v in test_features.items():
    print(f"  {k}: {v:.4f}")

## 4. G-Code Text Sampling

Sample lines from the beginning, middle, and end of each g-code file to capture
the overall structure for the text embedding branch.

In [ ]:
def sample_gcode_text(filepath: str, n_lines: int = TEXT_SAMPLE_LINES) -> str:
    """Sample lines from beginning, middle, and end of a g-code file."""
    with open(filepath, "r", errors="ignore") as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    if len(lines) <= n_lines:
        return "\n".join(lines)

    third = n_lines // 3
    remainder = n_lines - 2 * third

    begin = lines[:third]
    mid_start = (len(lines) - third) // 2
    middle = lines[mid_start:mid_start + third]
    end = lines[-remainder:]

    sampled = begin + ["\n;--- MID SAMPLE ---"] + middle + ["\n;--- END SAMPLE ---"] + end
    return "\n".join(sampled)

# Test
sample_text = sample_gcode_text(str(DATA_DIR / "1.gcode"))
print(f"Sampled text length: {len(sample_text)} chars")
print(sample_text[:500])

## 5. Extract All Features & Embeddings

In [ ]:
from tqdm import tqdm

print("Extracting g-code features...")
all_features = []
all_texts = []
valid_indices = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        feats = extract_gcode_features(row["path"])
        text = sample_gcode_text(row["path"])
        all_features.append(feats)
        all_texts.append(text)
        valid_indices.append(idx)
    except Exception as e:
        print(f"Error processing {row['path']}: {e}")

df = df.loc[valid_indices].reset_index(drop=True)
feature_df = pd.DataFrame(all_features)
print(f"\nExtracted features for {len(feature_df)} files")
print(f"Feature columns ({len(feature_df.columns)}): {list(feature_df.columns)}")

In [ ]:
# Generate text embeddings
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=str(DEVICE))

print("Encoding g-code text samples...")
text_embeddings = embed_model.encode(
    all_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Text embedding shape: {text_embeddings.shape}")

## 6. Prepare Dataset

In [ ]:
# Normalize numerical features
scaler = StandardScaler()
feature_matrix = scaler.fit_transform(feature_df.values.astype(np.float32))
# Replace any NaN/inf from bad g-codes
feature_matrix = np.nan_to_num(feature_matrix, nan=0.0, posinf=0.0, neginf=0.0)

labels = df["label"].values

print(f"Feature matrix: {feature_matrix.shape}")
print(f"Text embeddings: {text_embeddings.shape}")
print(f"Labels: {labels.shape}, distribution: {np.bincount(labels)}")

In [ ]:
# Train/val split
(
    feat_train, feat_val,
    emb_train, emb_val,
    y_train, y_val,
    idx_train, idx_val,
) = train_test_split(
    feature_matrix, text_embeddings, labels, np.arange(len(labels)),
    test_size=VAL_SPLIT, random_state=RANDOM_SEED, stratify=labels,
)

print(f"Train: {len(y_train)} (pos={y_train.sum()}, neg={len(y_train)-y_train.sum()})")
print(f"Val:   {len(y_val)} (pos={y_val.sum()}, neg={len(y_val)-y_val.sum()})")

In [ ]:
class GCodeDataset(Dataset):
    def __init__(self, features, embeddings, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.embeddings[idx], self.labels[idx]

train_ds = GCodeDataset(feat_train, emb_train, y_train)
val_ds = GCodeDataset(feat_val, emb_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

## 7. Hybrid Classifier Model

Two branches:
- **Feature branch**: MLP on extracted g-code statistics
- **Embedding branch**: projection layer on text embeddings

Concatenated, then classification head.

In [ ]:
class HybridGCodeClassifier(nn.Module):
    def __init__(self, n_features: int, embedding_dim: int, hidden_dim: int = 128):
        super().__init__()

        # Feature branch
        self.feature_branch = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )

        # Embedding branch
        self.embedding_branch = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, features, embeddings):
        feat_out = self.feature_branch(features)
        emb_out = self.embedding_branch(embeddings)
        combined = torch.cat([feat_out, emb_out], dim=1)
        return self.classifier(combined).squeeze(-1)

    def get_embedding(self, features, embeddings):
        """Get the combined embedding (before classification head) for inference/similarity."""
        feat_out = self.feature_branch(features)
        emb_out = self.embedding_branch(embeddings)
        return torch.cat([feat_out, emb_out], dim=1)


n_features = feature_matrix.shape[1]
embedding_dim = text_embeddings.shape[1]
model = HybridGCodeClassifier(n_features, embedding_dim).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 8. Training Loop

In [ ]:
# Handle class imbalance with weighted loss
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
if n_neg > 0:
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
else:
    # All samples are positive (no negatives yet), use weight 1.0
    pos_weight = torch.tensor([1.0], dtype=torch.float32).to(DEVICE)
    print("WARNING: No negative samples yet. Training will not be meaningful")
    print("         until you add non-gun g-codes to ../data-prep/g-code-non-gun/")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
train_losses = []
val_losses = []
val_accs = []
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    for feat_batch, emb_batch, label_batch in train_loader:
        feat_batch = feat_batch.to(DEVICE)
        emb_batch = emb_batch.to(DEVICE)
        label_batch = label_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(feat_batch, emb_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(label_batch)

    train_loss = epoch_loss / len(train_ds)
    train_losses.append(train_loss)

    # Validate
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for feat_batch, emb_batch, label_batch in val_loader:
            feat_batch = feat_batch.to(DEVICE)
            emb_batch = emb_batch.to(DEVICE)
            label_batch = label_batch.to(DEVICE)

            logits = model(feat_batch, emb_batch)
            loss = criterion(logits, label_batch)
            val_loss += loss.item() * len(label_batch)

            preds = torch.sigmoid(logits) > 0.5
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label_batch.cpu().numpy())

    val_loss /= len(val_ds)
    val_losses.append(val_loss)
    val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
    val_accs.append(val_acc)

    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_hybrid_classifier.pt")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

## 9. Evaluation

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label="Train")
ax1.plot(val_losses, label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.set_title("Loss")

ax2.plot(val_accs, label="Val Acc", color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.set_title("Validation Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load("best_hybrid_classifier.pt", map_location=DEVICE))
model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for feat_batch, emb_batch, label_batch in val_loader:
        feat_batch = feat_batch.to(DEVICE)
        emb_batch = emb_batch.to(DEVICE)

        logits = model(feat_batch, emb_batch)
        probs = torch.sigmoid(logits)
        preds = probs > 0.5

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(label_batch.numpy())

all_preds = np.array(all_preds, dtype=int)
all_labels = np.array(all_labels, dtype=int)

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["non-gun", "gun"]))

# Confusion matrix
if len(np.unique(all_labels)) > 1:
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["non-gun", "gun"], yticklabels=["non-gun", "gun"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()

    print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")
else:
    print("Only one class present, add non-gun samples to compute meaningful metrics.")

## 10. Inference Helper

Classify a new g-code file.

In [ ]:
def classify_gcode(filepath: str, model, embed_model, scaler, threshold=0.5):
    """Classify a single g-code file. Returns (is_gun: bool, confidence: float)."""
    features = extract_gcode_features(filepath)
    feat_vec = scaler.transform(
        np.array([list(features.values())], dtype=np.float32)
    )
    feat_vec = np.nan_to_num(feat_vec, nan=0.0, posinf=0.0, neginf=0.0)

    text = sample_gcode_text(filepath)
    text_emb = embed_model.encode([text], convert_to_numpy=True)

    model.eval()
    with torch.no_grad():
        feat_t = torch.tensor(feat_vec, dtype=torch.float32).to(DEVICE)
        emb_t = torch.tensor(text_emb, dtype=torch.float32).to(DEVICE)
        logit = model(feat_t, emb_t)
        prob = torch.sigmoid(logit).item()

    return prob >= threshold, prob

# Test inference
test_file = str(DATA_DIR / "1.gcode")
is_gun, confidence = classify_gcode(test_file, model, embed_model, scaler)
print(f"File: {test_file}")
print(f"Prediction: {'GUN PART' if is_gun else 'NOT gun part'} (confidence: {confidence:.4f})")

## 11. Save Model & Artifacts

In [ ]:
import pickle

SAVE_DIR = Path("./saved_model")
SAVE_DIR.mkdir(exist_ok=True)

# Save classifier weights
torch.save(model.state_dict(), SAVE_DIR / "hybrid_classifier.pt")

# Save scaler
with open(SAVE_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save config
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "n_features": n_features,
    "embedding_dim": embedding_dim,
    "feature_columns": list(feature_df.columns),
    "text_sample_lines": TEXT_SAMPLE_LINES,
}
with open(SAVE_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Model and artifacts saved to {SAVE_DIR}")
print(f"Files: {list(SAVE_DIR.iterdir())}")

## 12. Feature Importance Analysis (Optional)

Quick look at which numerical features matter most.

In [ ]:
# Visualize feature distributions (useful when you have both classes)
if len(df["label"].unique()) > 1:
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()
    cols = feature_df.columns[:20]
    for i, col in enumerate(cols):
        gun_vals = feature_df.loc[df["label"] == 1, col]
        non_gun_vals = feature_df.loc[df["label"] == 0, col]
        axes[i].hist(gun_vals, alpha=0.5, label="gun", bins=30)
        axes[i].hist(non_gun_vals, alpha=0.5, label="non-gun", bins=30)
        axes[i].set_title(col, fontsize=9)
        axes[i].legend(fontsize=7)
    plt.tight_layout()
    plt.show()
else:
    print("Feature distribution analysis available once you add non-gun samples.")
    print("\nCurrent feature summary (gun parts only):")
    display(feature_df.describe().T)

---

## Next Steps

1. **Add non-gun g-codes** to `../data-prep/g-code-non-gun/` (any `.gcode` files will be auto-discovered with label=0)
2. **Re-run all cells** after adding negative samples
3. **Tune hyperparameters** (hidden_dim, learning rate, epochs, text sample size)
4. **Optional: Fine-tune the text embedding model** itself (currently frozen) using contrastive loss if the classifier plateau accuracy is not high enough
5. **Export for production**, the `classify_gcode()` function + saved artifacts are all you need